In [0]:


dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import json, sys, os

sys.path.append(os.path.abspath(".."))
from src.interview_graph import build_interview_graph


class InterviewerModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        self.graph = build_interview_graph()

    # ← type hints ajoutés sur model_input et le retour
    def predict(
        self,
        context: mlflow.pyfunc.PythonModelContext,
        model_input: pd.DataFrame,
    ) -> pd.DataFrame:

        payload = json.loads(model_input["payload"].iloc[0])
        action  = payload.get("action")

        if action == "generate":
            etat = {
                "sujet_entretien":        payload["sujet"],
                "nombre_questions":       payload["nb_questions"],
                "questions":              [],
                "reponses":               [],
                "contextes_rag":          [],
                "feedback_recruteur":     "",
                "passed_recruteur":       False,
                "verdict_juge":           "",
                "juge_est_daccord":       False,
                "iteration":              0,
                "historique_juge":        [],
                "notes_juge_independant": "",
                "ecart_total":            0,
            }
            result = self.graph.invoke(etat, config={"recursion_limit": 10})
            return pd.DataFrame([{
                "result": json.dumps({
                    "questions": result["questions"],
                    "etat":      result,
                })
            }])

        elif action == "evaluate":
            etat_eval = {
                **payload["etat"],
                "reponses":         payload["reponses"],
                "juge_est_daccord": False,
            }
            result = self.graph.invoke(etat_eval, config={"recursion_limit": 10})
            return pd.DataFrame([{
                "result": json.dumps({
                    "feedback_recruteur": result["feedback_recruteur"],
                    "passed_recruteur":   result["passed_recruteur"],
                    "verdict_juge":       result["verdict_juge"],
                    "juge_est_daccord":   result["juge_est_daccord"],
                    "iteration":          result["iteration"],
                    "historique_juge":    result["historique_juge"],
                    "contextes_rag":      result["contextes_rag"],
                })
            }])

        else:
            raise ValueError(f"Action inconnue : {action}")

In [0]:
# Cellule 3 — Enregistrement dans MLflow
EXPERIMENT_NAME = "/Users/itssoufianetafahi@gmail.com/ai-interviewer"
mlflow.set_experiment(EXPERIMENT_NAME)

# Créer un input_example et output_example pour inférer la signature
input_example = pd.DataFrame([{
    "payload": json.dumps({
        "action": "generate",
        "sujet": "Python basics",
        "nb_questions": 3
    })
}])

output_example = pd.DataFrame([{
    "result": json.dumps({
        "questions": ["Q1", "Q2", "Q3"],
        "etat": {}
    })
}])

signature = mlflow.models.infer_signature(input_example, output_example)

with mlflow.start_run(run_name="interviewer-v1") as run:
    mlflow.pyfunc.log_model(
        artifact_path="interviewer",
        python_model=InterviewerModel(),
        input_example=input_example,
        signature=signature,
         code_paths=[
            "../src/",      # ← contient interview_graph.py
            "../config/",   # ← contient settings.py
        ],
        pip_requirements=[
            "langgraph",
            "langchain-core",
            "langchain-community",
            "langchain-huggingface",
            "databricks-langchain",     # ← ajouter cette ligne
            "faiss-cpu",
            "sentence-transformers",
            "databricks-vectorsearch",
        ],
        registered_model_name="ai-interviewer",
    )
    print(f"Run ID     : {run.info.run_id}")
    print(f"Modèle URI : runs:/{run.info.run_id}/interviewer")
    print("Modèle enregistré dans MLflow Registry !")

In [0]:
# Cellule 4 — Définir un alias "champion" sur la dernière version
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "workspace.default.ai-interviewer"

# Récupérer toutes les versions et prendre la dernière
versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max([int(v.version) for v in versions])

# Définir l'alias "champion" sur cette version
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=latest_version
)

print(f"Alias 'champion' défini sur version {latest_version}")

In [0]:
# Cellule optionnelle — inspecter l'artifact après log
import mlflow

client = mlflow.tracking.MlflowClient()
run_id = run.info.run_id

artifacts = client.list_artifacts(run_id, "interviewer")
for a in artifacts:
    print(a.path)

# Vous devriez voir :
# interviewer/code/src/interview_graph.py
# interviewer/code/config/settings.py
# interviewer/python_model.pkl
# interviewer/requirements.txt